# Метод k ближайших соседей: прогноз по похожим наблюдениям

**Проект «ИИ для учёных».** Практический блокнот к странице алгоритма «Метод k ближайших соседей».

## Что делает метод

Метод k ближайших соседей предсказывает ответ для нового объекта по наиболее похожим на него наблюдениям из обучающей выборки. Это ленивый алгоритм: обучение сводится к запоминанию выборки, а вся работа происходит при предсказании — поиске k ближайших соседей по выбранной метрике расстояния и агрегировании их ответов.

Метод применим, когда близкие по признакам объекты ожидаемо имеют близкий ответ, а явную модель строить нежелательно или сложно. В этом блокноте мы подберём число соседей k, оценим качество и визуализируем границы решения. Расчётное время прохождения — около пяти минут.

## Интуиция за методом

Представьте, что вы попали в незнакомый город и хотите понять, безопасный ли район. Самый простой способ — посмотреть на 5 ближайших улиц и оценить, что там происходит. Если 4 из 5 ближайших улиц безопасные — скорее всего, и вы в безопасном месте. Именно так рассуждает метод k ближайших соседей: чтобы классифицировать новый объект, он находит k наиболее похожих объектов в обучающей выборке и принимает решение «голосованием».

Удивительная особенность метода: он не строит никакой явной модели. Весь «интеллект» — это хранение обучающей выборки и умение искать в ней похожие объекты. Именно поэтому метод называют «ленивым» — он откладывает всю работу до момента предсказания.

Число k — ключевой параметр: маленькое k (k=1) означает «смотри только на самого похожего соседа» — чувствительно к шуму. Большое k — «смотри на много соседей» — усредняет, может пропустить локальную структуру.

**Ключевые термины:**
- **Метрика расстояния** — правило измерения «похожести» объектов; чаще всего евклидово расстояние (обычное «прямое»). Если признаки имеют разный масштаб, расстояние будет несправедливо зависеть от самых крупных по значению признаков.
- **Стандартизация** — приведение всех признаков к единому масштабу (среднее = 0, разброс = 1). Обязательна для kNN.
- **Кросс-валидация** — метод подбора гиперпараметра k: данные многократно делятся на обучение/проверку, качество усредняется.
- **Граница решения** — линия (или поверхность) в пространстве признаков, разделяющая области, куда модель относит объекты разных классов.
- **Главные компоненты** — новые оси координат, вдоль которых данные варьируют максимально; используются для визуализации в 2D.

## 1. Установка библиотек

В среде Google Colab перечисленные библиотеки, как правило, уже установлены. Ячейка ниже гарантирует их наличие, в том числе при локальном запуске.

In [ ]:
%pip install -q scikit-learn numpy pandas matplotlib

## 2. Единый стиль графиков

Все графики в блокнотах проекта оформляются в едином визуальном стиле сайта «ИИ для учёных»: общая палитра, шрифты, толщины линий и сетка. Ниже встроено содержимое стилевого шаблона `notebooks/viz_style.py` — отдельный файл загружать не нужно. Вызов `apply_viz_style()` настраивает matplotlib один раз на весь блокнот.

In [ ]:
# Единый стилевой шаблон графиков проекта «ИИ для учёных».
# Значения синхронизированы с токенами темы сайта (styles/tokens.css).
VIZ_TOKENS = {
    "background": "#ffffff",  # фон полотна
    "node_fill":  "#eef1f6",  # заливка блоков
    "node_text":  "#0e1726",  # основной текст
    "edge":       "#4d5e78",  # линии, оси, подписи
    "grid":       "#dce2ec",  # сетка координат
    "series":     ["#2563eb", "#0d9488", "#b45309", "#4d5e78"],  # цвета рядов
}
VIZ = VIZ_TOKENS


def apply_viz_style():
    """Настраивает matplotlib под единый стиль сайта."""
    import matplotlib as mpl
    from cycler import cycler
    t = VIZ_TOKENS
    mpl.rcParams.update({
        "font.family": "sans-serif",
        "font.sans-serif": ["Segoe UI", "DejaVu Sans", "Arial", "Helvetica"],
        "font.size": 12, "axes.titlesize": 15, "axes.titleweight": "semibold",
        "axes.labelsize": 13, "xtick.labelsize": 11, "ytick.labelsize": 11,
        "legend.fontsize": 11,
        "figure.facecolor": t["background"], "axes.facecolor": t["background"],
        "savefig.facecolor": t["background"], "figure.dpi": 110, "savefig.dpi": 150,
        "axes.prop_cycle": cycler(color=t["series"]),
        "text.color": t["node_text"], "axes.labelcolor": t["node_text"],
        "axes.titlecolor": t["node_text"], "axes.edgecolor": t["edge"],
        "xtick.color": t["edge"], "ytick.color": t["edge"], "axes.linewidth": 1.2,
        "axes.grid": True, "grid.color": t["grid"], "grid.linewidth": 1.0,
        "grid.linestyle": "-", "axes.axisbelow": True,
        "lines.linewidth": 2.0, "lines.markersize": 6, "patch.linewidth": 1.5,
        "axes.spines.top": False, "axes.spines.right": False,
        "legend.frameon": False,
    })


def get_palette(n=None):
    """Возвращает список цветов рядов из токенов темы."""
    series = VIZ_TOKENS["series"]
    if n is None:
        return list(series)
    return [series[i % len(series)] for i in range(n)]


apply_viz_style()
print("Единый стиль графиков подключён.")

## 3. Демонстрационные данные

Используем открытый набор по биохимическим характеристикам вина (`wine` из `scikit-learn`): 178 образцов, 13 числовых признаков, три сорта. Задача — отнести образец к сорту по измеренным характеристикам.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_wine

data = load_wine(as_frame=True)
X = data.data
y = data.target
feature_names = list(X.columns)
class_names = list(data.target_names)

print(f'Наблюдений: {X.shape[0]}, признаков: {X.shape[1]}')
print(f'Классы: {class_names}')
X.head()

## 4. Применение метода

Метод kNN опирается на расстояния, поэтому признаки обязательно стандартизуются — иначе признак с большими числовыми значениями будет несправедливо доминировать над остальными. Соберём конвейер: сначала стандартизация, затем kNN. Пока используем k=5 как начальное значение, а в следующем шаге подберём оптимальное k.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y)

model = make_pipeline(StandardScaler(),
                      KNeighborsClassifier(n_neighbors=5))
model.fit(X_train, y_train)
print('Модель построена; обучающая выборка запомнена.')

### Подбор числа соседей k

Параметр k — это **гиперпараметр**: его нельзя выучить из данных напрямую, нужно перебирать вручную. Правило: маленькое k реагирует на каждую точку и «запоминает» случайные выбросы (переобучение), большое k сглаживает всё и может пропустить реальную структуру (недообучение). Подберём оптимальное k кросс-валидацией.

In [ ]:
from sklearn.model_selection import cross_val_score
import matplotlib.pyplot as plt

k_values = range(1, 26)
cv_scores = []
for k in k_values:
    pipe = make_pipeline(StandardScaler(),
                         KNeighborsClassifier(n_neighbors=k))
    cv_scores.append(cross_val_score(pipe, X_train, y_train,
                                     cv=5).mean())

best_k = list(k_values)[int(np.argmax(cv_scores))]
fig, ax = plt.subplots(figsize=(9.5, 5.2))
ax.plot(list(k_values), cv_scores, marker='o',
        color=VIZ['series'][0])
ax.axvline(best_k, color=VIZ['series'][2], linestyle='--',
           label=f'лучшее k = {best_k}')
ax.set_xlabel('Число соседей k')
ax.set_ylabel('Доля верных ответов (кросс-валидация)')
ax.set_title('Подбор числа соседей')
ax.legend()
fig.tight_layout()
plt.show()

**Как читать этот график.** По горизонтали — число соседей k. По вертикали — доля верных ответов на кросс-валидации. Кривая обычно имеет «горб»: слева от вершины — переобучение (k слишком мало, модель шумит), справа — недообучение (k слишком велико, граница слишком размыта). Оранжевая пунктирная линия отмечает лучшее k, которое и будет использовано в дальнейшем.

### Качество классификации

Обучим модель с подобранным k и оценим её на отложенной тестовой выборке.

In [ ]:
from sklearn.metrics import accuracy_score, classification_report

model = make_pipeline(StandardScaler(),
                      KNeighborsClassifier(n_neighbors=best_k))
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
print(f'Доля верных ответов на тесте: {accuracy_score(y_test, y_pred):.3f}\n')
print(classification_report(y_test, y_pred, target_names=class_names))

### Визуализация границ решения

Чтобы увидеть, как kNN делит пространство признаков, спроецируем данные на две главные компоненты (PCA — метод снижения размерности, который сохраняет максимум информации в 2D) и построим границы решения для двух значений k.

**Зачем PCA?** У нас 13 признаков, нарисовать их все нельзя. PCA выбирает два «направления» в пространстве признаков, вдоль которых данные варьируют больше всего, и проецирует на них.

In [ ]:
from sklearn.decomposition import PCA
from matplotlib.colors import ListedColormap

scaler = StandardScaler().fit(X_train)
pca = PCA(n_components=2).fit(scaler.transform(X_train))
X2 = pca.transform(scaler.transform(X_train))

h = 0.05
xx, yy = np.meshgrid(
    np.arange(X2[:, 0].min() - 1, X2[:, 0].max() + 1, h),
    np.arange(X2[:, 1].min() - 1, X2[:, 1].max() + 1, h))
cmap_bg = ListedColormap(['#dfe9fb', '#d6efec', '#f4e4cf'])
cmap_pt = [VIZ['series'][0], VIZ['series'][1], VIZ['series'][2]]

fig, axes = plt.subplots(1, 2, figsize=(13.5, 5.4))
for ax, k in zip(axes, [1, best_k]):
    clf = KNeighborsClassifier(n_neighbors=k).fit(X2, y_train)
    Z = clf.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    ax.contourf(xx, yy, Z, cmap=cmap_bg, alpha=0.8)
    for cls in np.unique(y_train):
        m = y_train == cls
        ax.scatter(X2[m, 0], X2[m, 1], color=cmap_pt[cls],
                   edgecolor='white', s=35,
                   label=class_names[cls])
    ax.set_title(f'Границы решения, k = {k}')
    ax.set_xlabel('Главная компонента 1')
    ax.set_ylabel('Главная компонента 2')
    ax.grid(False)
axes[0].legend(loc='best')
fig.tight_layout()
plt.show()

**Как читать эти графики.** Цвет фона — класс, которому модель присваивает точку в данной области. Точки — реальные наблюдения обучающей выборки. Левый график (k=1): граница очень «изрезанная» — каждая точка «захватывает» свою территорию, модель реагирует на шум и запоминает случайные выбросы (переобучение). Правый график (подобранное k): граница плавная и устойчивая — смотрим на нескольких соседей одновременно. Сравните: правильно подобранное k даёт более разумную и обобщаемую границу.

## 5. Интерпретация результата

- **Кривая подбора k**: вершина кривой — оптимальное число соседей. Слева от вершины модель переобучена (реагирует на шум), справа — недообучена (граница чрезмерно сглажена).
- **Доля верных ответов и отчёт по классам**: оценивают качество на наблюдениях, которых модель не видела.
- **Границы решения**: при k = 1 граница изрезана и плотно облегает отдельные точки; при подобранном k она глаже и устойчивее к шуму.

Метод k ближайших соседей требует хранить всю обучающую выборку и медленно предсказывает на больших данных; он также чувствителен к неинформативным признакам — отбор и масштабирование признаков критичны.

## 6. Попробуйте на своих данных

Замените демонстрационный набор своими данными для задачи классификации.

1. Загрузите файл в Colab через вкладку «Файлы».
2. Снимите комментарии в ячейке ниже и укажите имя файла и столбец с меткой класса.
3. Выполните ячейки раздела 4.

## 7. Поэкспериментируйте сами

1. **Уберите стандартизацию.** Удалите `StandardScaler()` из конвейера в разделе 4 и повторите подбор k. Как меняется качество? Посмотрите на границы решения — они, вероятно, станут несимметричными.

2. **Попробуйте другую метрику расстояния.** В `KNeighborsClassifier` задайте параметр `metric='manhattan'` (манхэттенское расстояние — сумма абсолютных разностей). Сравните качество с евклидовой метрикой.

3. **Добавьте шум в признаки.** Добавьте к матрице X несколько столбцов случайного шума: `X_noisy = X.copy(); X_noisy['noise1'] = np.random.randn(len(X))`. Как неинформативные признаки влияют на качество kNN? Это демонстрирует чувствительность метода к «мусорным» признакам.

In [ ]:
# --- Шаблон загрузки своих данных ---
# df = pd.read_csv('my_data.csv')
# target_column = 'класс'
#
# y = df[target_column].astype('category').cat.codes.to_numpy()
# X = df.drop(columns=[target_column]).select_dtypes('number')
# class_names = [str(c) for c in
#                df[target_column].astype('category').cat.categories]
#
# print(f'Наблюдений: {X.shape[0]}, признаков: {X.shape[1]}')
# Далее повторите ячейки раздела 4.

## 8. Краткие выводы

- kNN не строит явной модели: он запоминает обучающую выборку и классифицирует новые объекты по голосованию k ближайших соседей.
- Стандартизация признаков обязательна: без неё признак с большим масштабом будет несправедливо доминировать в вычислении расстояния.
- Число соседей k — ключевой гиперпараметр: k=1 переобучает, большое k сглаживает. Оптимальное k подбирается кросс-валидацией.
- Граница решения при k=1 изрезана и плотно облегает точки, при оптимальном k — плавная и обобщаемая.
- Слабые стороны: медленное предсказание на больших данных, чувствительность к неинформативным признакам, высокое потребление памяти (нужно хранить все данные).

## Три вопроса для самопроверки

Ответьте до того, как раскроете подсказку, — это проверяет, что метод понят, а не просто запущен.

1. На графике подбора k кривая кросс-валидации имеет чёткий максимум при k = 7, а затем монотонно снижается до k = 25. Что происходит с границей решения при дальнейшем увеличении k, и почему точность падает?
2. Вы убрали `StandardScaler` из конвейера и обнаружили, что качество резко упало, а граница решения стала несимметричной. Какой признак из набора `wine` скорее всего «захватил» всё расстояние и почему?
3. Ваш набор данных содержит 500 наблюдений и 200 числовых признаков (например, масс-спектрометрические пики). Объясните, почему kNN, скорее всего, окажется неэффективным, и назовите один подготовительный шаг, способный улучшить ситуацию.

<details>
<summary>Показать ориентиры для ответов</summary>

1. При больших k граница решения становится всё более сглаженной: модель усредняет большую окрестность и перестаёт разрешать локальную структуру данных. В пределе при k = n модель всегда предсказывает самый частый класс во всей выборке — это недообучение.
2. Признак `proline` (концентрация пролина) имеет значения порядка 400–1600, тогда как большинство остальных — в диапазоне 0–5. Евклидово расстояние практически определяется только разностью значений пролина; остальные 12 признаков вносят пренебрежимый вклад.
3. В 200-мерном пространстве евклидово расстояние теряет смысл: все точки становятся примерно равноудалены друг от друга (проклятие размерности). Первый шаг — снижение размерности (PCA, UMAP) или отбор наиболее информативных признаков, чтобы уменьшить число измерений до единиц–десятков.
</details>
